# LeetCode 642 — Design Search Autocomplete System bằng Python

- Link: https://leetcode.com/problems/design-search-autocomplete-system/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Thiết kế lớp (Design), Trie (cây tiền tố), Sắp xếp theo nhiều khóa, Heap, Xử lý chuỗi

> **Ghi chú Premium:** đây là bài dành cho tài khoản LeetCode Premium. Phần đề bài dưới đây được viết lại theo hiểu biết về bài (chữ ký lớp, ví dụ, ràng buộc). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại** chi tiết đề (đặc biệt: luật xếp hạng khi hòa số lần, và giới hạn số lời gọi `input`) trước khi nộp.

Skeleton trên LeetCode (Python3):

```python
class AutocompleteSystem:

    def __init__(self, sentences: List[str], times: List[int]):

    def input(self, c: str) -> List[str]:

```

## 1. Giới thiệu & mục tiêu học

Bạn gõ vào ô tìm kiếm của Google, sau mỗi ký tự lại hiện ra vài gợi ý "hot" nhất. Bài này yêu cầu xây dựng đúng hệ thống đó ở quy mô nhỏ. Đây là bài **thiết kế** điển hình: dữ liệu thay đổi theo thời gian (câu mới được thêm), truy vấn tới liên tục theo từng ký tự. Cấu trúc trung tâm là **Trie** — cây tiền tố.

Sau notebook này bạn sẽ:

1. Hiểu đúng giao diện: `input(c)` nhận **từng ký tự**, `'#'` kết thúc câu và ghi vào lịch sử.
2. Sắp xếp theo **hai khóa**: số lần giảm dần, rồi thứ tự ASCII tăng dần — bằng khóa tuple `(-count, sentence)`.
3. Viết lời giải vét cạn (quét mọi câu) và bản lọc dần ứng viên.
4. Cài đặt **Trie** với con trỏ "nút hiện tại" để không phải đi lại từ gốc, rồi nâng cấp: **mỗi nút lưu sẵn top-3**.
5. Kiểm chứng mọi cách bằng chuỗi thao tác ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Thiết kế lớp `AutocompleteSystem`:

- `AutocompleteSystem(sentences, times)`: lịch sử ban đầu — câu `sentences[i]` đã được gõ `times[i]` lần.
- `input(c)`:
  - Nếu `c` là chữ cái thường hoặc dấu cách: thêm `c` vào **câu đang gõ** (tiền tố hiện tại). Trả về **tối đa 3** câu trong lịch sử **bắt đầu bằng** tiền tố hiện tại, xếp theo:
    1. **số lần** (độ "hot") giảm dần;
    2. nếu bằng nhau, thứ tự **ASCII** tăng dần (dấu cách `' '` có mã 32, nhỏ hơn chữ cái).
  - Nếu `c == '#'`: câu đang gõ kết thúc — **cộng 1** vào số lần của câu đó trong lịch sử (câu mới thì thêm vào với số lần 1), đặt lại câu đang gõ về rỗng, trả về `[]`.

**Ví dụ**

```text
AutocompleteSystem(["i love you", "island", "iroman", "i love leetcode"], [5, 3, 2, 2])

input('i') -> ["i love you", "island", "i love leetcode"]
   bốn câu đều bắt đầu bằng "i". "i love you" (5), "island" (3);
   "iroman" và "i love leetcode" cùng 2 lần -> so ASCII: ' ' (32) < 'r' (114)
   nên "i love leetcode" đứng trước "iroman". Lấy 3 câu đầu.
input(' ') -> ["i love you", "i love leetcode"]
input('a') -> []          không câu nào bắt đầu bằng "i a"
input('#') -> []          "i a" được thêm vào lịch sử với số lần 1
```

**Ràng buộc (theo hiểu biết về bài)**

- `n == sentences.length == times.length`, `1 <= n <= 100`
- `1 <= sentences[i].length <= 100`, `1 <= times[i] <= 50`
- `c` là chữ cái thường, `'#'` hoặc `' '`.
- Mỗi câu được gõ kết thúc bằng `'#'`, dài trong khoảng `[1, 200]`.
- Tối đa `5000` lời gọi `input`.

In [1]:
sentences = ["i love you", "island", "iroman", "i love leetcode"]
times = [5, 3, 2, 2]

# Tính tay câu trả lời cho input('i') bằng sorted với khóa tuple
prefix = "i"
cands = [(s, t) for s, t in zip(sentences, times) if s.startswith(prefix)]
print(sorted(cands, key=lambda st: (-st[1], st[0])))
print("top 3:", [s for s, _ in sorted(cands, key=lambda st: (-st[1], st[0]))[:3]])
print("ord(' ') =", ord(' '), "| ord('r') =", ord('r'))

[('i love you', 5), ('island', 3), ('i love leetcode', 2), ('iroman', 2)]
top 3: ['i love you', 'island', 'i love leetcode']
ord(' ') = 32 | ord('r') = 114


## 3. Kiến thức nền

### 3.1. Sắp xếp nhiều khóa bằng tuple, giảm dần bằng số đối

`sorted(items, key=lambda x: (-count, sentence))`: Python so sánh tuple từ trái sang phải → số lần lớn đứng trước (vì `-count` nhỏ hơn), hòa thì chuỗi nhỏ (ASCII) đứng trước.

In [2]:
items = [("b", 3), ("a", 3), ("c", 5), ("a b", 3)]
print(sorted(items, key=lambda x: (-x[1], x[0])))

[('c', 5), ('a', 3), ('a b', 3), ('b', 3)]


### 3.2. So sánh chuỗi theo ASCII trong Python

`str` so sánh từng ký tự theo mã Unicode (với chữ thường và dấu cách trùng ASCII). Chuỗi là tiền tố của chuỗi kia thì ngắn hơn đứng trước.

In [3]:
print("i love leetcode" < "iroman", "ab" < "abc", "a b" < "ab")
print(sorted(["iroman", "i love leetcode", "island", "i"]))

True True True
['i', 'i love leetcode', 'iroman', 'island']


### 3.3. `str.startswith` và `dict.get`

`s.startswith(p)` kiểm tra tiền tố trong `O(len(p))`. `d.get(k, 0) + 1` để cộng dồn số lần, kể cả khi khóa chưa có.

In [4]:
hist = {"island": 3}
for s in ["island", "iron"]:
    hist[s] = hist.get(s, 0) + 1
print(hist, "island".startswith("isl"), "island".startswith("ir"))

{'island': 4, 'iron': 1} True False


### 3.4. Trie — cây tiền tố

Mỗi nút là một dict `ký tự → nút con`. Đi từ gốc theo các ký tự của tiền tố, ta tới nút đại diện cho tiền tố đó; **mọi câu trong cây con** của nút đều có tiền tố này. Đánh dấu cuối câu bằng một trường lưu số lần.

In [5]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0          # > 0 nếu có câu kết thúc tại đây

root = TrieNode()
for s, t in [("ab", 2), ("abc", 1), ("b", 4)]:
    node = root
    for ch in s:
        node = node.children.setdefault(ch, TrieNode())
    node.count += t

def collect(node, path, out):
    if node.count:
        out.append((path, node.count))
    for ch, child in node.children.items():
        collect(child, path + ch, out)

out = []
collect(root.children['a'], "a", out)
print("các câu có tiền tố 'a':", out)

các câu có tiền tố 'a': [('ab', 2), ('abc', 1)]


### 3.5. `heapq.nsmallest` — lấy `k` phần tử nhỏ nhất

Khi chỉ cần top-3 trong `m` ứng viên, `heapq.nsmallest(3, items, key=...)` chạy `O(m log 3)` thay vì sắp xếp toàn bộ `O(m log m)`.

In [6]:
import heapq
pairs = [("i love you", 5), ("island", 3), ("iroman", 2), ("i love leetcode", 2)]
print(heapq.nsmallest(3, pairs, key=lambda st: (-st[1], st[0])))

[('i love you', 5), ('island', 3), ('i love leetcode', 2)]


### 3.6. Trạng thái bên trong đối tượng giữa các lần gọi

`input` được gọi **nhiều lần liên tiếp**; đối tượng phải nhớ "câu đang gõ" (và với Trie: "nút hiện tại") giữa các lần gọi — lưu vào thuộc tính `self.…`.

In [7]:
class Typer:
    def __init__(self):
        self.buf = []
    def type(self, c):
        if c == '#':
            done, self.buf = ''.join(self.buf), []
            return f"xong: {done!r}"
        self.buf.append(c)
        return ''.join(self.buf)

t = Typer()
print([t.type(c) for c in "hi#a"])

['h', 'hi', "xong: 'hi'", 'a']


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Mỗi `input` là một truy vấn "top-3 theo tiền tố"

Sau khi gõ `"i "`, câu hỏi là: trong lịch sử, những câu nào bắt đầu bằng `"i "`? Xếp theo `(-count, câu)`, lấy 3 câu đầu. `'#'` là một thao tác **cập nhật** lịch sử.

### 4.2. Tiền tố chỉ dài thêm cho tới `'#'`

Trong một câu đang gõ, tiền tố **chỉ dài thêm** từng ký tự. Tập câu khớp tiền tố mới là **tập con** của tập khớp tiền tố cũ → có thể **lọc dần** thay vì quét lại từ đầu, hoặc trong Trie **đi tiếp một bước** từ nút hiện tại.

Khi tiền tố không còn khớp câu nào (rơi khỏi cây), mọi ký tự sau đó đều trả `[]` cho tới `'#'`.

### 4.3. Tính tay tiếp ví dụ sau `'#'`

Sau ví dụ, lịch sử có thêm `"i a": 1`. Gõ tiếp:

- `input('i')` → các câu có tiền tố `"i"`: `i love you (5)`, `island (3)`, `i love leetcode (2)`, `iroman (2)`, `i a (1)` → top 3: `["i love you", "island", "i love leetcode"]`.
- `input(' ')` → `i love you (5)`, `i love leetcode (2)`, `i a (1)` → `["i love you", "i love leetcode", "i a"]`.
- `input('a')` → `["i a"]`.
- `input('#')` → `[]`, `"i a"` lên 2 lần.

In [8]:
hist = dict(zip(sentences, times))
hist["i a"] = 1
for p in ["i", "i ", "i a"]:
    cands = sorted((s for s in hist if s.startswith(p)), key=lambda s: (-hist[s], s))
    print(f"{p!r:>6} -> {cands[:3]}")

   'i' -> ['i love you', 'island', 'i love leetcode']
  'i ' -> ['i love you', 'i love leetcode', 'i a']
 'i a' -> ['i a']


## 5. Cách 1 — Brute Force: quét mọi câu ở mỗi ký tự

**Ý tưởng.** Lưu `hist: dict câu → số lần` và `cur: list ký tự đang gõ`. Mỗi `input(c)` (không phải `'#'`): tiền tố `p = ''.join(cur)`; lọc mọi câu `startswith(p)`, sắp theo `(-count, câu)`, trả 3 câu đầu.

In [9]:
from typing import List

class AutocompleteBrute:
    def __init__(self, sentences: List[str], times: List[int]):
        self.hist = {}
        for s, t in zip(sentences, times):
            self.hist[s] = self.hist.get(s, 0) + t
        self.cur = []

    def input(self, c: str) -> List[str]:
        if c == '#':
            s = ''.join(self.cur)
            self.hist[s] = self.hist.get(s, 0) + 1
            self.cur = []
            return []
        self.cur.append(c)
        p = ''.join(self.cur)
        cands = [s for s in self.hist if s.startswith(p)]
        cands.sort(key=lambda s: (-self.hist[s], s))
        return cands[:3]

def run(system, chars):
    return [system.input(c) for c in chars]

ac = AutocompleteBrute(sentences, times)
res = run(ac, "i a#")
print(res)
assert res == [["i love you", "island", "i love leetcode"], ["i love you", "i love leetcode"], [], []]

[['i love you', 'island', 'i love leetcode'], ['i love you', 'i love leetcode'], [], []]


**Độ phức tạp.** Mỗi ký tự: `O(N · L)` để lọc (`N` câu, `L` độ dài tiền tố) + `O(M log M)` sắp `M` ứng viên. Với `N` tới `100 + 5000` câu (mỗi `'#'` có thể thêm câu mới) và 5000 lời gọi: cỡ `5000 · 5100 · L` — chậm nhưng thực tế vẫn có thể qua vì dữ liệu nhỏ. Space `O(tổng độ dài câu)`.

**Nút thắt.** (1) Dựng lại chuỗi `p` và quét **mọi** câu ở mỗi ký tự, dù tiền tố chỉ dài thêm 1. (2) Sắp xếp toàn bộ ứng viên chỉ để lấy 3.

In [10]:
import time, random
random.seed(0)
def rand_sentence(alpha="ab ", lo=1, hi=8):
    return ''.join(random.choice(alpha) for _ in range(random.randint(lo, hi))).strip() or "a"

big_sent = list({rand_sentence("abcde ", 3, 30) for _ in range(3000)})
big_times = [random.randint(1, 50) for _ in big_sent]
typing = ''.join(random.choice(big_sent)[:random.randint(1, 10)] + '#' for _ in range(400))
t = time.perf_counter()
run(AutocompleteBrute(big_sent, big_times), typing)
print(f"Brute: {len(typing)} lời gọi trên {len(big_sent)} câu: {time.perf_counter() - t:.3f}s")

Brute: 2522 lời gọi trên 2944 câu: 0.245s


## 6. Bước đột phá

**Insight 1 (lọc dần).** Tập ứng viên của tiền tố dài hơn ⊂ tập của tiền tố ngắn hơn. Giữ danh sách ứng viên, mỗi ký tự chỉ lọc lại **trên danh sách đó** và chỉ cần so **một ký tự** (vị trí `len(p) - 1`).

**Insight 2 (Trie + con trỏ).** Trie gom các câu theo tiền tố. Giữ con trỏ `self.node` tới nút của tiền tố hiện tại; mỗi ký tự chỉ **đi một bước** `node = node.children.get(c)`. Ứng viên = mọi câu trong cây con.

**Insight 3 (top-3 cache tại mỗi nút).** Vì chỉ hỏi top-3 và số lần **chỉ tăng**, mỗi nút có thể lưu sẵn danh sách top-3 của cây con. Khi một câu tăng số lần, chỉ các nút **trên đường đi** của câu đó cần cập nhật. Truy vấn thành `O(1)`.

**Invariant (Cách 4):** với mọi nút `v`, `v.top` là 3 câu đứng đầu (theo `(-count, câu)`) trong các câu thuộc cây con của `v`.

**Quy tắc nếu…thì…:**
- Nếu `c == '#'` thì cập nhật lịch sử, đặt lại tiền tố và con trỏ về gốc, trả `[]`.
- Nếu con trỏ đã `None` (rơi khỏi cây) thì trả `[]` và giữ `None` cho tới `'#'`.
- Nếu một câu tăng số lần thì nó chỉ có thể **đi lên** trong các top-3 trên đường đi của nó; câu khác không thể tự đi lên.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | `input` (ký tự thường) | `input('#')` |
|---|---|---|---|
| 1 | Quét mọi câu (mục 5) | `O(N·L + M log M)` | `O(L)` |
| 2 | Lọc dần ứng viên | `O(M_cũ + M log M)` | `O(L)` |
| 3 | Trie + con trỏ + DFS cây con | `O(kích thước cây con + M log 3)` | `O(L)` |
| 4 | Trie, mỗi nút lưu top-3 | `O(1)` | `O(L)` |

### 7.1. Cách 2 — Lọc dần ứng viên

**Giải thích.** Thêm `self.cands`: danh sách câu khớp tiền tố hiện tại. Ký tự đầu tiên của câu đang gõ: lọc từ toàn bộ `hist`. Ký tự sau: chỉ giữ câu trong `self.cands` có `s[i] == c` (với `i` = vị trí ký tự mới). Dùng `heapq.nsmallest` để lấy top-3.

In [11]:
class AutocompleteFilter:
    def __init__(self, sentences: List[str], times: List[int]):
        self.hist = {}
        for s, t in zip(sentences, times):
            self.hist[s] = self.hist.get(s, 0) + t
        self.cur = []
        self.cands = None                  # None = chưa lọc (tiền tố rỗng)

    def input(self, c: str) -> List[str]:
        if c == '#':
            s = ''.join(self.cur)
            self.hist[s] = self.hist.get(s, 0) + 1
            self.cur, self.cands = [], None
            return []
        i = len(self.cur)                  # vị trí của ký tự mới
        self.cur.append(c)
        pool = self.hist.keys() if self.cands is None else self.cands
        self.cands = [s for s in pool if len(s) > i and s[i] == c]
        return heapq.nsmallest(3, self.cands, key=lambda s: (-self.hist[s], s))

ac = AutocompleteFilter(sentences, times)
res = run(ac, "i a#i a#")
print(res)
assert res[:4] == [["i love you", "island", "i love leetcode"], ["i love you", "i love leetcode"], [], []]

[['i love you', 'island', 'i love leetcode'], ['i love you', 'i love leetcode'], [], [], ['i love you', 'island', 'i love leetcode'], ['i love you', 'i love leetcode', 'i a'], ['i a'], []]


**Đọc code:**

- `self.cands is None` nghĩa là đang ở ký tự đầu của câu mới → lọc trên toàn bộ lịch sử.
- Điều kiện `len(s) > i and s[i] == c`: câu đủ dài và khớp ký tự mới; các ký tự trước đã khớp từ những lần lọc trước.
- `heapq.nsmallest(3, ...)` thay cho sắp xếp toàn bộ.
- Khi `'#'`: lịch sử thay đổi nên reset `cands = None` (lần sau lọc lại từ đầu — số lần mới được đọc trực tiếp từ `hist`).

**Code nộp LeetCode (Cách 2):**

In [12]:
class AutocompleteSystemFilter:          # trên LeetCode đổi tên thành AutocompleteSystem
    def __init__(self, sentences: List[str], times: List[int]):
        self.hist = {}
        for s, t in zip(sentences, times):
            self.hist[s] = self.hist.get(s, 0) + t
        self.cur = []
        self.cands = None

    def input(self, c: str) -> List[str]:
        if c == '#':
            s = ''.join(self.cur)
            self.hist[s] = self.hist.get(s, 0) + 1
            self.cur, self.cands = [], None
            return []
        i = len(self.cur)
        self.cur.append(c)
        pool = self.hist if self.cands is None else self.cands
        self.cands = [s for s in pool if len(s) > i and s[i] == c]
        hist = self.hist
        return heapq.nsmallest(3, self.cands, key=lambda s: (-hist[s], s))

assert run(AutocompleteSystemFilter(sentences, times), "i a#") == run(AutocompleteBrute(sentences, times), "i a#")
print("AutocompleteSystemFilter OK")

AutocompleteSystemFilter OK


**Độ phức tạp Cách 2:** ký tự đầu `O(N)`; các ký tự sau `O(M_cũ)` với `M_cũ` = số ứng viên trước đó (giảm nhanh); top-3 `O(M log 3)`. `'#'`: `O(L)`.

### 7.2. Cách 3 — Trie + con trỏ + DFS cây con

**Giải thích.** Xây Trie từ lịch sử; nút cuối câu lưu `count` và chính câu đó (`sentence`) để khỏi dựng lại chuỗi khi DFS. Giữ:

- `self.node`: nút của tiền tố hiện tại (gốc khi bắt đầu câu; `None` nếu rơi khỏi cây).
- `self.cur`: các ký tự đã gõ (để biết câu khi gặp `'#'`).

Mỗi ký tự: đi một bước; nếu còn nút thì DFS cây con thu `(count, sentence)`, lấy top-3 bằng `nsmallest`.

In [13]:
class Node:
    __slots__ = ("children", "count", "sentence", "top")
    def __init__(self):
        self.children = {}
        self.count = 0
        self.sentence = None
        self.top = []            # dùng cho Cách 4

class AutocompleteTrie:
    def __init__(self, sentences: List[str], times: List[int]):
        self.root = Node()
        for s, t in zip(sentences, times):
            self._add(s, t)
        self.cur = []
        self.node = self.root

    def _add(self, s, t):
        node = self.root
        for ch in s:
            node = node.children.setdefault(ch, Node())
        node.count += t
        node.sentence = s

    def _collect(self, node):
        out, stack = [], [node]
        while stack:
            v = stack.pop()
            if v.count:
                out.append((-v.count, v.sentence))
            stack.extend(v.children.values())
        return out

    def input(self, c: str) -> List[str]:
        if c == '#':
            self._add(''.join(self.cur), 1)
            self.cur, self.node = [], self.root
            return []
        self.cur.append(c)
        if self.node is not None:
            self.node = self.node.children.get(c)
        if self.node is None:
            return []
        return [s for _, s in heapq.nsmallest(3, self._collect(self.node))]

ac = AutocompleteTrie(sentences, times)
res = run(ac, "i a#i a#")
print(res)
assert res == run(AutocompleteBrute(sentences, times), "i a#i a#")

[['i love you', 'island', 'i love leetcode'], ['i love you', 'i love leetcode'], [], [], ['i love you', 'island', 'i love leetcode'], ['i love you', 'i love leetcode', 'i a'], ['i a'], []]


**Đọc code:**

- `_add` dùng `setdefault` để tạo nút con nếu chưa có; cộng `t` vào `count` (hỗ trợ cả câu trùng trong input ban đầu).
- `_collect` dùng **stack** (DFS không đệ quy) để tránh giới hạn đệ quy với câu dài 200.
- Phần tử `(-count, sentence)`: so sánh tuple tự nhiên đúng luật xếp hạng → `nsmallest(3, ...)` không cần `key`.
- Khi rơi khỏi cây, `self.node = None` và giữ nguyên tới `'#'`.

**Code nộp LeetCode (Cách 3):**

In [14]:
class AutocompleteSystemTrie:            # trên LeetCode đổi tên thành AutocompleteSystem
    def __init__(self, sentences: List[str], times: List[int]):
        self.root = {}
        for s, t in zip(sentences, times):
            self._add(s, t)
        self.cur = []
        self.node = self.root

    def _add(self, s, t):
        node = self.root
        for ch in s:
            node = node.setdefault(ch, {})
        node['#'] = node.get('#', 0) + t       # khóa '#' lưu số lần ('#' không xuất hiện trong câu)

    def input(self, c: str) -> List[str]:
        if c == '#':
            self._add(''.join(self.cur), 1)
            self.cur, self.node = [], self.root
            return []
        self.cur.append(c)
        if self.node is not None:
            self.node = self.node.get(c)
        if self.node is None:
            return []
        prefix = ''.join(self.cur)
        out, stack = [], [(self.node, prefix)]
        while stack:
            v, path = stack.pop()
            for ch, child in v.items():
                if ch == '#':
                    out.append((-child, path))
                else:
                    stack.append((child, path + ch))
        return [s for _, s in heapq.nsmallest(3, out)]

assert run(AutocompleteSystemTrie(sentences, times), "i a#i a#") == run(AutocompleteBrute(sentences, times), "i a#i a#")
print("AutocompleteSystemTrie OK")

AutocompleteSystemTrie OK


Bản nộp dùng **dict lồng nhau** thay cho lớp `Node` — một mẹo gọn: khóa đặc biệt `'#'` (không bao giờ là ký tự của câu) lưu số lần.

**Độ phức tạp Cách 3:** mỗi ký tự `O(S + M log 3)` với `S` = số nút trong cây con (tối đa tổng độ dài các câu khớp). `'#'`: `O(L)`. Space: `O(tổng độ dài câu)`.

### 7.3. Cách 4 — Trie, mỗi nút lưu top-3

**Giải thích.** Mỗi nút giữ `top`: danh sách tối đa 3 cặp `(-count, sentence)` đã sắp, đại diện cho cây con. Khi câu `s` có số lần mới `cnt`:

```text
với mỗi nút v trên đường đi của s (kể cả nút cuối):
    bỏ s khỏi v.top nếu có
    thêm (-cnt, s), sắp xếp, giữ 3 phần tử đầu
```

Truy vấn: đi một bước rồi trả `[s for _, s in node.top]`.

Vì sao đúng? Số lần chỉ **tăng**. Một câu `x ≠ s` không đổi số lần nên không thể "nhảy vào" top-3 của nút nào; chỉ `s` có thể đi lên — và ta cập nhật đúng các nút chứa `s` trong cây con.

In [15]:
class AutocompleteTop3:
    def __init__(self, sentences: List[str], times: List[int]):
        self.root = Node()
        self.count = {}
        for s, t in zip(sentences, times):
            self._bump(s, t)
        self.cur = []
        self.node = self.root

    def _bump(self, s, t):
        cnt = self.count.get(s, 0) + t
        self.count[s] = cnt
        item = (-cnt, s)
        node = self.root
        for ch in s:
            node = node.children.setdefault(ch, Node())
            top = [x for x in node.top if x[1] != s]     # bỏ bản ghi cũ của s
            top.append(item)
            top.sort()
            node.top = top[:3]

    def input(self, c: str) -> List[str]:
        if c == '#':
            self._bump(''.join(self.cur), 1)
            self.cur, self.node = [], self.root
            return []
        self.cur.append(c)
        if self.node is not None:
            self.node = self.node.children.get(c)
        if self.node is None:
            return []
        return [s for _, s in self.node.top]

ac = AutocompleteTop3(sentences, times)
res = run(ac, "i a#i a#")
print(res)
assert res == run(AutocompleteBrute(sentences, times), "i a#i a#")

[['i love you', 'island', 'i love leetcode'], ['i love you', 'i love leetcode'], [], [], ['i love you', 'island', 'i love leetcode'], ['i love you', 'i love leetcode', 'i a'], ['i a'], []]


**Đọc code:**

- `self.count` (dict) giữ tổng số lần mỗi câu — cần để biết số lần **mới** khi `'#'`.
- Vòng `for ch in s`: cập nhật `top` của mọi nút trên đường đi (gốc không cần vì tiền tố rỗng không bao giờ được hỏi).
- `top` có tối đa 4 phần tử trước khi cắt → sắp xếp là `O(1)`.

**Code nộp LeetCode (Cách 4)** — xem mục 9 (lời giải chính).

**Độ phức tạp Cách 4:** `input` ký tự thường **`O(1)`**; `'#'` và khởi tạo `O(L)` mỗi câu (mỗi nút `O(1)` vì top ≤ 4 phần tử, so sánh chuỗi tốn thêm `O(L)` trong trường hợp xấu). Space `O(tổng độ dài câu)` nút, mỗi nút thêm 3 tham chiếu.

## 8. TRACE từng bước

Trace ví dụ đề rồi gõ lại `"i a#"` lần nữa, in tiền tố, nút hiện tại có tồn tại không, và kết quả. Với Cách 4 in thêm `top` của nút.

| Lời gọi | Tiền tố | Nút | Kết quả |
|---|---|---|---|
| `'i'` | `"i"` | có | `["i love you", "island", "i love leetcode"]` |
| `' '` | `"i "` | có | `["i love you", "i love leetcode"]` |
| `'a'` | `"i a"` | **không** (rơi khỏi cây) | `[]` |
| `'#'` | — | về gốc | `[]`, thêm `"i a": 1` |
| `'i'` | `"i"` | có | `["i love you", "island", "i love leetcode"]` |
| `' '` | `"i "` | có | `["i love you", "i love leetcode", "i a"]` |
| `'a'` | `"i a"` | có | `["i a"]` |
| `'#'` | — | về gốc | `[]`, `"i a": 2` |

In [16]:
def trace_top3(sentences, times, chars):
    ac = AutocompleteTop3(sentences, times)
    for c in chars:
        res = ac.input(c)
        prefix = ''.join(ac.cur)
        if c == '#':
            print(f"input('#') -> {res}   | lịch sử: { {k: v for k, v in ac.count.items() if k.startswith('i a')} }")
        else:
            state = "None (rơi khỏi cây)" if ac.node is None else f"top={ac.node.top}"
            print(f"input({c!r}) tiền tố={prefix!r:<7} -> {res}\n      nút: {state}")
    return ac

_ = trace_top3(sentences, times, "i a#i a#")

input('i') tiền tố='i'     -> ['i love you', 'island', 'i love leetcode']
      nút: top=[(-5, 'i love you'), (-3, 'island'), (-2, 'i love leetcode')]
input(' ') tiền tố='i '    -> ['i love you', 'i love leetcode']
      nút: top=[(-5, 'i love you'), (-2, 'i love leetcode')]
input('a') tiền tố='i a'   -> []
      nút: None (rơi khỏi cây)
input('#') -> []   | lịch sử: {'i a': 1}
input('i') tiền tố='i'     -> ['i love you', 'island', 'i love leetcode']
      nút: top=[(-5, 'i love you'), (-3, 'island'), (-2, 'i love leetcode')]
input(' ') tiền tố='i '    -> ['i love you', 'i love leetcode', 'i a']
      nút: top=[(-5, 'i love you'), (-2, 'i love leetcode'), (-1, 'i a')]
input('a') tiền tố='i a'   -> ['i a']
      nút: top=[(-1, 'i a')]
input('#') -> []   | lịch sử: {'i a': 2}


Trace thêm Cách 2 để thấy danh sách ứng viên **co lại** qua từng ký tự:

In [17]:
def trace_filter(sentences, times, chars):
    ac = AutocompleteFilter(sentences, times)
    for c in chars:
        res = ac.input(c)
        if c == '#':
            print("input('#') -> [] (reset ứng viên)")
        else:
            print(f"input({c!r}) tiền tố={''.join(ac.cur)!r:<8} ứng viên={ac.cands} -> {res}")

trace_filter(sentences, times, "i lo#")

input('i') tiền tố='i'      ứng viên=['i love you', 'island', 'iroman', 'i love leetcode'] -> ['i love you', 'island', 'i love leetcode']
input(' ') tiền tố='i '     ứng viên=['i love you', 'i love leetcode'] -> ['i love you', 'i love leetcode']
input('l') tiền tố='i l'    ứng viên=['i love you', 'i love leetcode'] -> ['i love you', 'i love leetcode']
input('o') tiền tố='i lo'   ứng viên=['i love you', 'i love leetcode'] -> ['i love you', 'i love leetcode']
input('#') -> [] (reset ứng viên)


## 9. Code nộp LeetCode

Lời giải chính: **Trie với top-3 lưu sẵn tại mỗi nút** (Cách 4).

In [18]:
from typing import List

class TrieNode3:
    __slots__ = ("children", "top")
    def __init__(self):
        self.children = {}
        self.top = []                     # tối đa 3 cặp (-count, sentence), đã sắp

class AutocompleteSystem:
    def __init__(self, sentences: List[str], times: List[int]):
        self.root = TrieNode3()
        self.count = {}                   # câu -> tổng số lần
        for s, t in zip(sentences, times):
            self._bump(s, t)
        self.cur = []                     # các ký tự của câu đang gõ
        self.node = self.root             # nút của tiền tố hiện tại (None nếu rơi khỏi cây)

    def _bump(self, s: str, t: int) -> None:
        cnt = self.count.get(s, 0) + t
        self.count[s] = cnt
        item = (-cnt, s)
        node = self.root
        for ch in s:
            node = node.children.setdefault(ch, TrieNode3())
            top = [x for x in node.top if x[1] != s]
            top.append(item)
            top.sort()
            node.top = top[:3]

    def input(self, c: str) -> List[str]:
        if c == '#':
            self._bump(''.join(self.cur), 1)
            self.cur = []
            self.node = self.root
            return []
        self.cur.append(c)
        if self.node is not None:
            self.node = self.node.children.get(c)
        if self.node is None:
            return []
        return [s for _, s in self.node.top]

sys_ = AutocompleteSystem(["i love you", "island", "iroman", "i love leetcode"], [5, 3, 2, 2])
assert sys_.input('i') == ["i love you", "island", "i love leetcode"]
assert sys_.input(' ') == ["i love you", "i love leetcode"]
assert sys_.input('a') == []
assert sys_.input('#') == []
print("AutocompleteSystem OK")

AutocompleteSystem OK


## 10. Vì sao thuật toán đúng

**Cách 1** cài đặt nguyên văn đặc tả: lọc theo tiền tố, sắp theo `(-count, câu)`, lấy 3.

**Cách 2:** Bất biến: sau khi gõ `i + 1` ký tự của câu hiện tại, `cands` = mọi câu trong lịch sử có tiền tố đúng bằng `i + 1` ký tự đó. Cơ sở: ký tự đầu lọc trên toàn bộ lịch sử theo `s[0] == c`. Bước: câu có tiền tố `p + c` ⇔ có tiền tố `p` (thuộc `cands` cũ) và `s[i] == c`. Lịch sử chỉ đổi ở `'#'`, lúc đó `cands` được reset.

**Cách 3:** Trong Trie, các câu có tiền tố `p` chính là các câu kết thúc trong cây con của nút ứng với `p`; nếu không có nút đó thì không có câu nào. Con trỏ `self.node` luôn là nút của tiền tố hiện tại (đi đúng một cạnh mỗi ký tự), hoặc `None`.

**Cách 4 — invariant:** `v.top` = 3 phần tử nhỏ nhất (theo `(-count, câu)`) trong tập câu thuộc cây con của `v`.

- Khi câu `s` tăng số lần, chỉ các nút trên đường đi của `s` có tập câu chứa `s`; các nút khác không đổi.
- Với nút `v` trên đường đi: thứ hạng của các câu khác **không đổi**; `s` chỉ đi **lên** (số lần tăng). Gọi `T` là top-3 cũ. Top-3 mới nằm trong `(T \ {s}) ∪ {s mới}`: một câu `x ∉ T, x ≠ s` trước đó đã đứng sau 3 câu của `T`; sau cập nhật, nó vẫn đứng sau `T \ {s}` và sau `s mới` (vì `s mới` tốt hơn `s cũ`) — tức vẫn đứng sau ít nhất 3 câu. Vậy chỉ cần bỏ `s` cũ, thêm `s` mới, sắp lại, cắt 3.

Điểm mấu chốt: lập luận trên **cần số lần chỉ tăng**. Nếu bài cho phép giảm (xóa câu), cache top-3 có thể thiếu ứng viên (xem mục 15).

## 11. Phân tích Time/Space

Gọi `N` = số câu trong lịch sử, `L` = độ dài câu (≤ 200), `S` = kích thước cây con, `M` = số ứng viên.

| Cách | ký tự thường | `'#'` | Khởi tạo | Space |
|---|---|---|---|---|
| 1. Quét | `O(N·L + M log M)` | `O(L)` | `O(N)` | `O(N·L)` |
| 2. Lọc dần | `O(M_cũ + M)` | `O(L)` | `O(N)` | `O(N·L)` |
| 3. Trie + DFS | `O(S)` | `O(L)` | `O(N·L)` | `O(N·L)` |
| 4. Trie + top-3 | `O(1)` | `O(L · L)`* | `O(N · L · L)`* | `O(N·L)` |

\* so sánh chuỗi trong `sort` tốn tới `O(L)`; thực tế rất nhỏ.

In [19]:
for cls in (AutocompleteBrute, AutocompleteFilter, AutocompleteTrie, AutocompleteTop3, AutocompleteSystem):
    t = time.perf_counter()
    out = run(cls(big_sent, big_times), typing)
    print(f"{cls.__name__:>20}: {time.perf_counter() - t:.3f}s")

   AutocompleteBrute: 0.221s
  AutocompleteFilter: 0.086s


    AutocompleteTrie: 0.309s
    AutocompleteTop3: 0.063s
  AutocompleteSystem: 0.029s


## 12. Bộ test

Gồm: ví dụ đề, edge case (câu trùng trong input ban đầu, hòa số lần → ASCII, dấu cách nhỏ hơn chữ cái, gõ câu đã có làm thay đổi thứ hạng, rơi khỏi cây rồi `'#'`, câu là tiền tố của câu khác) và chuỗi thao tác ngẫu nhiên có seed so với brute force.

In [20]:
CLASSES = [AutocompleteBrute, AutocompleteFilter, AutocompleteSystemFilter, AutocompleteTrie,
           AutocompleteSystemTrie, AutocompleteTop3, AutocompleteSystem]

def check_all(sentences, times, chars, expected=None):
    outs = [run(cls(sentences, times), chars) for cls in CLASSES]
    for cls, o in zip(CLASSES, outs):
        assert o == outs[0], (cls.__name__, chars, o, outs[0])
    if expected is not None:
        assert outs[0] == expected, (outs[0], expected)
    return outs[0]

# Ví dụ đề
check_all(sentences, times, "i a#", [["i love you", "island", "i love leetcode"], ["i love you", "i love leetcode"], [], []])

# Hòa số lần -> ASCII; dấu cách < chữ cái
check_all(["ab", "a b", "abc"], [1, 1, 1], "a", [["a b", "ab", "abc"]])

# Câu là tiền tố của câu khác
check_all(["a", "ab"], [1, 1], "ab#", [["a", "ab"], ["ab"], []])

# Gõ câu đã có -> tăng hot, đổi thứ hạng
check_all(["b", "ba"], [1, 1], "ba#b", [["b", "ba"], ["ba"], [], ["ba", "b"]])

# Câu trùng trong input ban đầu (được cộng dồn)
check_all(["x", "y", "x"], [1, 1, 1], "x", [["x"]])

# Rơi khỏi cây rồi '#', sau đó câu mới xuất hiện trong gợi ý
check_all(["abc"], [3], "zz#z", [[], [], [], ["zz"]])

# Nhiều hơn 3 ứng viên
check_all(["aa", "ab", "ac", "ad"], [1, 2, 3, 4], "a", [["ad", "ac", "ab"]])
print("Test cố định OK")

Test cố định OK


In [21]:
rng = random.Random(2024)
for trial in range(300):
    n = rng.randint(1, 8)
    sents = [''.join(rng.choice("ab ") for _ in range(rng.randint(1, 5))) for _ in range(n)]
    sents = [s if s.strip() else "a" for s in sents]
    tms = [rng.randint(1, 4) for _ in range(n)]
    chars = []
    for _ in range(rng.randint(1, 6)):
        base = rng.choice(sents) if rng.random() < 0.6 else ''.join(rng.choice("abc ") for _ in range(rng.randint(1, 4)))
        chars.extend(base[:rng.randint(1, len(base))])
        if rng.random() < 0.85:
            chars.append('#')
    check_all(sents, tms, chars)
print("300 kịch bản ngẫu nhiên: mọi cách khớp brute force")

300 kịch bản ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Sắp xếp sai chiều khi hòa (`-s` không dùng được với chuỗi)

**Code sai:** `sorted(cands, key=lambda s: (hist[s], s), reverse=True)` — đảo chiều **cả hai** khóa: số lần giảm dần đúng, nhưng khi hòa lại xếp ASCII **giảm dần** (sai).

In [22]:
hist_demo = {"iroman": 2, "i love leetcode": 2, "island": 3}
wrong = sorted(hist_demo, key=lambda s: (hist_demo[s], s), reverse=True)
right = sorted(hist_demo, key=lambda s: (-hist_demo[s], s))
print("sai :", wrong)
print("đúng:", right)

sai : ['island', 'iroman', 'i love leetcode']
đúng: ['island', 'i love leetcode', 'iroman']


**Code đúng:** đảo dấu **chỉ** khóa số: `(-count, sentence)` và không dùng `reverse`.

### Lỗi 2: Quên trả `[]` và giữ trạng thái khi rơi khỏi cây

**Code sai:** khi `node.children.get(c)` là `None`, lần sau lại tìm từ gốc hoặc gây `AttributeError` (`None.children`).

In [23]:
class TrieFallWrong(AutocompleteTrie):
    def input(self, c):
        if c == '#':
            return super().input(c)
        self.cur.append(c)
        nxt = self.node.children.get(c) if self.node is not None else None
        if nxt is None:
            self.node = self.root          # SAI: quay về gốc thay vì giữ None
            return []
        self.node = nxt
        return [s for _, s in heapq.nsmallest(3, self._collect(self.node))]

print("sai :", run(TrieFallWrong(["ab", "b"], [1, 1]), "zb"))
print("đúng:", run(AutocompleteSystem(["ab", "b"], [1, 1]), "zb"))

sai : [[], ['b']]
đúng: [[], []]


Sau khi gõ `"z"`, tiền tố là `"zb"` — không câu nào khớp. Bản sai quay về gốc nên coi như tiền tố `"b"`.

**Code đúng:** giữ `self.node = None` cho tới `'#'`.

### Lỗi 3: Không ghi lại câu ở `'#'` khi câu đó rơi khỏi cây

**Code sai:** chỉ tăng số lần nếu `self.node` còn tồn tại (vì "nút cuối câu" là `self.node`).

**Vì sao sai:** câu mới hoàn toàn (không có trong Trie) phải được **thêm** vào. Luôn chèn lại từ gốc theo `self.cur`.

In [24]:
class TrieRecordWrong(AutocompleteTrie):
    def input(self, c):
        if c == '#':
            if self.node is not None:          # SAI: bỏ qua câu mới
                self.node.count += 1
                self.node.sentence = ''.join(self.cur)
            self.cur, self.node = [], self.root
            return []
        return super().input(c)

print("sai :", run(TrieRecordWrong(["abc"], [3]), "zz#z"))
print("đúng:", run(AutocompleteSystem(["abc"], [3]), "zz#z"))

sai : [[], [], [], []]
đúng: [[], [], [], ['zz']]


**Code đúng:** `self._add(''.join(self.cur), 1)` — chèn từ gốc.

### Lỗi 4: Cache top-3 nhưng không bỏ bản ghi cũ của câu

**Code sai:** chỉ `append((-cnt, s))` rồi cắt 3 → câu xuất hiện **hai lần** trong top.

In [25]:
class Top3DupWrong(AutocompleteTop3):
    def _bump(self, s, t):
        cnt = self.count.get(s, 0) + t
        self.count[s] = cnt
        node = self.root
        for ch in s:
            node = node.children.setdefault(ch, Node())
            node.top = sorted(node.top + [(-cnt, s)])[:3]      # SAI: không bỏ bản cũ

print("sai :", run(Top3DupWrong(["a", "ab"], [1, 1]), "a#a"))
print("đúng:", run(AutocompleteSystem(["a", "ab"], [1, 1]), "a#a"))

sai : [['a', 'ab'], [], ['a', 'a', 'ab']]
đúng: [['a', 'ab'], [], ['a', 'ab']]


**Code đúng:** lọc bỏ `x[1] == s` trước khi thêm.

## 14. Mẹo & nhận dạng pattern

- **"Gợi ý theo tiền tố"**, **"tìm mọi từ bắt đầu bằng…"** → Trie.
- Truy vấn **theo từng ký tự liên tiếp** → giữ **con trỏ nút hiện tại**, không đi lại từ gốc.
- Chỉ cần **top-k** và trọng số **chỉ tăng** → lưu sẵn top-k ở mỗi nút, cập nhật dọc đường đi.
- Sắp nhiều khóa ngược chiều nhau → đảo dấu khóa số, giữ nguyên khóa chuỗi.
- Top-k nhỏ trong nhiều ứng viên → `heapq.nsmallest(k, ...)`.
- Bài cùng họ: 208 (Implement Trie), 1268 (Search Suggestions System), 211 (Add and Search Word), 745 (Prefix and Suffix Search).

## 15. Biến thể

1. **Search Suggestions System (LC 1268)** — không có số lần, gợi ý 3 sản phẩm nhỏ nhất theo từ điển cho mỗi tiền tố.
2. **Cho phép xóa câu / giảm số lần** — cache top-3 không còn đúng (câu thứ 4 có thể phải đi lên); dùng Cách 3 (DFS) hoặc lưu top-k lớn hơn / cấu trúc cân bằng.
3. **Top-k tùy ý** thay cho 3 — đổi hằng số; Cách 4 cập nhật `O(k log k)` mỗi nút.
4. **Gợi ý theo tiền tố của từ cuối** (không phải cả câu) — tách theo dấu cách.

In [26]:
def suggested_products(products, search_word):
    """Biến thể 1 (LC 1268): sắp xếp + lọc dần."""
    products = sorted(products)
    res, cands = [], products
    for i, ch in enumerate(search_word):
        cands = [p for p in cands if len(p) > i and p[i] == ch]
        res.append(cands[:3])
    return res

out = suggested_products(["mobile", "mouse", "moneypot", "monitor", "mousepad"], "mouse")
print(out)
assert out[0] == ["mobile", "moneypot", "monitor"] and out[-1] == ["mouse", "mousepad"]

[['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], ['mouse', 'mousepad'], ['mouse', 'mousepad']]


In [27]:
class AutocompleteK(AutocompleteTop3):
    """Biến thể 3: top-k tùy ý."""
    def __init__(self, sentences, times, k=3):
        self.k = k
        super().__init__(sentences, times)

    def _bump(self, s, t):
        cnt = self.count.get(s, 0) + t
        self.count[s] = cnt
        node = self.root
        for ch in s:
            node = node.children.setdefault(ch, Node())
            node.top = sorted([x for x in node.top if x[1] != s] + [(-cnt, s)])[:self.k]

ac = AutocompleteK(sentences, times, k=5)
print(ac.input('i'))
assert ac.input(' ') == ["i love you", "i love leetcode"]

['i love you', 'island', 'i love leetcode', 'iroman']


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | ký tự thường | `'#'` | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét | lọc mọi câu, sắp, lấy 3 | `O(N·L + M log M)` | `O(L)` | trọng tài, phỏng vấn mở đầu |
| 2. Lọc dần | ứng viên co lại theo tiền tố | `O(M_cũ)` | `O(L)` | ngắn gọn, đủ nhanh với dữ liệu nhỏ |
| 3. Trie + DFS | con trỏ nút, DFS cây con | `O(S)` | `O(L)` | khi cho phép xóa/giảm số lần |
| 4. Trie + top-3 | cache top-3 tại mỗi nút | `O(1)` | `O(L)` | truy vấn nhiều, số lần chỉ tăng |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code, với `AutocompleteSystem(["abc", "abd", "ab"], [2, 2, 1])`: kết quả của `input('a')`, `input('b')`, `input('d')`, `input('#')`, `input('a')`?

**Bài 2 (tính tay).** Với hệ ở Bài 1 sau các lời gọi trên, viết `top` của nút ứng với tiền tố `"ab"` (Cách 4).

**Bài 3 (điền TODO).** Hoàn thiện `rank(hist, prefix)` trả về top-3 theo đúng luật (dùng `sorted` hoặc `nsmallest`).

**Bài 4 (tự viết lại).** Viết lại Cách 3 `MyAutocomplete` (Trie dict lồng nhau + DFS).

**Bài 5 (mini challenge).** Viết `AutocompleteWithDelete` hỗ trợ thêm `delete(sentence)` (xóa câu khỏi lịch sử). Gợi ý: Cách 3 vẫn đúng nếu đặt số lần về 0. Kiểm tra bằng thao tác ngẫu nhiên so với một brute force có `delete`.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def rank(hist, prefix):
    # TODO: lọc câu startswith(prefix), sắp theo (-hist[s], s), lấy 3
    pass

# Bài 4 — TODO
class MyAutocomplete:
    def __init__(self, sentences, times):
        # TODO: root = {}; chèn mọi câu; cur = []; node = root
        pass
    def input(self, c):
        # TODO
        pass

# Bài 5 — TODO
class AutocompleteWithDelete:
    def __init__(self, sentences, times):
        pass
    def input(self, c):
        pass
    def delete(self, sentence):
        # TODO: đặt số lần của câu về 0 (nút '#' = 0 hoặc xóa khóa)
        pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `"abc"` và `"abd"` cùng 2 lần → ASCII `'c' < 'd'`.
- Mức 2: `'a'` → `["abc", "abd", "ab"]`; `'b'` → như cũ; `'d'` → `["abd"]`.
- Mức 3: `'#'` làm `"abd"` lên 3; `'a'` → `["abd", "abc", "ab"]`.

**Bài 2.**
- Mức 1: nút `"ab"` chứa cả ba câu.
- Mức 2: số lần sau cập nhật: `abd 3`, `abc 2`, `ab 1`.
- Mức 3: `top = [(-3, "abd"), (-2, "abc"), (-1, "ab")]`.

**Bài 3.**
- Mức 1: `cands = [s for s in hist if s.startswith(prefix)]`.
- Mức 2: `key=lambda s: (-hist[s], s)`.
- Mức 3: `[:3]`.

**Bài 4.**
- Mức 1: dùng khóa `'#'` trong dict lưu số lần.
- Mức 2: DFS bằng stack các cặp `(nút, đường đi)`.
- Mức 3: nhớ trường hợp `node is None`.

**Bài 5.**
- Mức 1: lấy Cách 3; `delete` đi theo câu từ gốc.
- Mức 2: nếu tới được nút cuối và có khóa `'#'`, xóa khóa đó.
- Mức 3: brute force có `delete` chỉ cần `hist.pop(s, None)`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
ac = AutocompleteSystem(["abc", "abd", "ab"], [2, 2, 1])
got = [ac.input(c) for c in "abd#a"]
print(got)
assert got == [["abc", "abd", "ab"], ["abc", "abd", "ab"], ["abd"], [], ["abd", "abc", "ab"]]

# Bài 2
node_ab = ac.root.children['a'].children['b']
print("top của 'ab':", node_ab.top)
assert node_ab.top == [(-3, "abd"), (-2, "abc"), (-1, "ab")]

# Bài 3
def rank_ans(hist, prefix):
    return sorted((s for s in hist if s.startswith(prefix)), key=lambda s: (-hist[s], s))[:3]

assert rank_ans(dict(zip(sentences, times)), "i") == ["i love you", "island", "i love leetcode"]
print("Bài 1–3 OK")

[['abc', 'abd', 'ab'], ['abc', 'abd', 'ab'], ['abd'], [], ['abd', 'abc', 'ab']]
top của 'ab': [(-3, 'abd'), (-2, 'abc'), (-1, 'ab')]
Bài 1–3 OK


In [30]:
# Bài 4
class MyAutocompleteAns:
    def __init__(self, sentences, times):
        self.root = {}
        for s, t in zip(sentences, times):
            self._add(s, t)
        self.cur, self.node = [], self.root

    def _add(self, s, t):
        node = self.root
        for ch in s:
            node = node.setdefault(ch, {})
        node['#'] = node.get('#', 0) + t

    def input(self, c):
        if c == '#':
            self._add(''.join(self.cur), 1)
            self.cur, self.node = [], self.root
            return []
        self.cur.append(c)
        self.node = self.node.get(c) if self.node is not None else None
        if self.node is None:
            return []
        out, stack = [], [(self.node, ''.join(self.cur))]
        while stack:
            v, path = stack.pop()
            for ch, child in v.items():
                if ch == '#':
                    if child > 0:
                        out.append((-child, path))
                else:
                    stack.append((child, path + ch))
        return [s for _, s in heapq.nsmallest(3, out)]

# Bài 5
class AutocompleteWithDeleteAns(MyAutocompleteAns):
    def delete(self, sentence):
        node = self.root
        for ch in sentence:
            node = node.get(ch)
            if node is None:
                return
        node.pop('#', None)

class BruteWithDelete(AutocompleteBrute):
    def delete(self, sentence):
        self.hist.pop(sentence, None)

rng = random.Random(99)
for trial in range(200):
    sents = [''.join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 6))]
    tms = [rng.randint(1, 3) for _ in sents]
    a, b, c3 = AutocompleteWithDeleteAns(sents, tms), BruteWithDelete(sents, tms), MyAutocompleteAns(sents, tms)
    d = AutocompleteBrute(sents, tms)
    for _ in range(25):
        if rng.random() < 0.15 and not a.cur:
            s = rng.choice(sents)
            a.delete(s); b.delete(s)
            c3 = None                       # sau delete không so với bản không hỗ trợ delete
            continue
        ch = rng.choice("ab#")
        ra, rb = a.input(ch), b.input(ch)
        assert ra == rb, (sents, ch, ra, rb)
        if c3 is not None:
            assert c3.input(ch) == d.input(ch)
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Trie + top-3 tại mỗi nút:**

```python
def _bump(self, s, t):
    cnt = self.count.get(s, 0) + t; self.count[s] = cnt
    node = self.root
    for ch in s:
        node = node.children.setdefault(ch, TrieNode3())
        top = [x for x in node.top if x[1] != s] + [(-cnt, s)]
        top.sort(); node.top = top[:3]

def input(self, c):
    if c == '#':
        self._bump(''.join(self.cur), 1); self.cur = []; self.node = self.root; return []
    self.cur.append(c)
    self.node = self.node.children.get(c) if self.node else None
    return [s for _, s in self.node.top] if self.node else []
```

**Checklist:**

- [ ] Khóa xếp hạng `(-count, sentence)`, **không** dùng `reverse=True`.
- [ ] `'#'`: ghi câu (kể cả câu mới, câu rơi khỏi cây), reset tiền tố **và** con trỏ.
- [ ] Rơi khỏi cây: trả `[]` và giữ `None` tới `'#'`.
- [ ] Câu trùng trong input ban đầu: cộng dồn số lần.
- [ ] Cache top-3: bỏ bản ghi cũ của câu trước khi thêm bản mới.
- [ ] Tên lớp/method đúng LeetCode: `AutocompleteSystem`, `input`.